# Most Complete Original Patient Overview

This notebook searches the original PSV files and displays the patient with the highest proportion of recorded measurements. It does not modify the dataset or production pipeline.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
files = sorted((RAW_DIR / 'training_setA').glob('*.psv')) + sorted((RAW_DIR / 'training_setB').glob('*.psv'))
print('Original patient files:', len(files))

Original patient files: 40336


## Find the patient with the most complete recorded values

The raw dataset is naturally sparse because tests are not recorded at every ICU hour. This cell selects the most complete available patient without filling or changing any values.

In [2]:
if not files:
    raise FileNotFoundError(f'No PSV files found under {RAW_DIR}')

rows = []
for path in files:
    frame = pd.read_csv(path, sep='|')
    rows.append({
        'file': str(path),
        'patient_id': path.stem,
        'rows': len(frame),
        'missing_values': int(frame.isna().sum().sum()),
        'completeness_percent': round(100 * frame.notna().to_numpy().mean(), 2),
    })

completeness = pd.DataFrame(rows).sort_values(
    ['completeness_percent', 'missing_values'], ascending=[False, True]
).reset_index(drop=True)
selected_file = Path(completeness.loc[0, 'file'])
selected_patient = pd.read_csv(selected_file, sep='|')

print('Selected patient:', completeness.loc[0, 'patient_id'])
print('Completeness:', f"{completeness.loc[0, 'completeness_percent']:.2f}%")
print('Missing values:', completeness.loc[0, 'missing_values'])
display(completeness.head(10))

Selected patient: p011500
Completeness: 61.79%
Missing values: 188


,file,patient_id,rows,missing_values,completeness_percent
0,e:\Earlysepsispredicton\data\raw\training_setA...,p011500,12,188,61.79
1,e:\Earlysepsispredicton\data\raw\training_setA...,p019481,9,156,57.72
2,e:\Earlysepsispredicton\data\raw\training_setB...,p100559,50,939,54.20
3,e:\Earlysepsispredicton\data\raw\training_setA...,p015094,12,229,53.46
4,e:\Earlysepsispredicton\data\raw\training_setB...,p106485,9,173,53.12
5,e:\Earlysepsispredicton\data\raw\training_setA...,p001426,21,404,53.08
6,e:\Earlysepsispredicton\data\raw\training_setA...,p008040,12,231,53.05
7,e:\Earlysepsispredicton\data\raw\training_setA...,p014646,46,896,52.49
8,e:\Earlysepsispredicton\data\raw\training_setB...,p115735,12,234,52.44
9,e:\Earlysepsispredicton\data\raw\training_setA...,p012153,17,332,52.37


## Original observations for the selected patient

In [3]:
print('Rows and columns:', selected_patient.shape)
display(selected_patient.head(20))
if {'ICULOS', 'SepsisLabel'}.issubset(selected_patient.columns):
    display(selected_patient[['ICULOS', 'SepsisLabel']].head(20))

Rows and columns: (12, 41)


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,71.0,100.0,35.9,98.25,66.5,50.25,14,NaN,-8.5,22.0,...,8.5,255.0,280.0,60.56,0,0,1,-52.79,4,0
1,80.0,99.0,36.1,94.75,61.5,49.00,15,NaN,-8.5,22.0,...,8.5,NaN,NaN,60.56,0,0,1,-52.79,5,0
2,80.0,95.5,36.1,87.00,58.0,47.75,17,NaN,-7.5,NaN,...,NaN,NaN,NaN,60.56,0,0,1,-52.79,6,1
3,98.5,NaN,36.2,103.25,70.5,58.50,23,NaN,-4.5,NaN,...,NaN,NaN,NaN,60.56,0,0,1,-52.79,7,1
4,85.0,94.5,36.4,90.25,59.5,49.00,24,NaN,-4.0,NaN,...,NaN,NaN,NaN,60.56,0,0,1,-52.79,8,1
5,77.0,100.0,36.5,92.50,61.0,48.50,24,NaN,-4.0,NaN,...,NaN,NaN,NaN,60.56,0,0,1,-52.79,9,1
6,79.0,100.0,36.7,103.25,69.5,52.25,24,NaN,-3.5,NaN,...,NaN,NaN,NaN,60.56,0,0,1,-52.79,10,1
7,88.0,100.0,36.9,93.50,67.0,58.00,24,NaN,0.0,NaN,...,NaN,NaN,NaN,60.56,0,0,1,-52.79,11,1
8,81.0,100.0,37.2,145.00,95.0,73.50,20,NaN,1.0,NaN,...,NaN,NaN,NaN,60.56,0,0,1,-52.79,12,1
9,80.0,98.0,37.1,106.00,70.0,55.00,18,NaN,1.0,NaN,...,NaN,NaN,NaN,60.56,0,0,1,-52.79,13,1


,ICULOS,SepsisLabel
0,4,0
1,5,0
2,6,1
3,7,1
4,8,1
5,9,1
6,10,1
7,11,1
8,12,1
9,13,1


## Missing-value status for the selected patient

In [4]:
missing_summary = pd.DataFrame({
    'missing_count': selected_patient.isna().sum(),
    'missing_percent': (selected_patient.isna().mean() * 100).round(2),
}).sort_values('missing_count', ascending=False)
display(missing_summary)

,missing_count,missing_percent
EtCO2,12,100.00
TroponinI,12,100.00
Bilirubin_direct,12,100.00
Calcium,11,91.67
Phosphate,11,91.67
Fibrinogen,11,91.67
Creatinine,10,83.33
PTT,10,83.33
Magnesium,10,83.33
Bilirubin_total,10,83.33


The selected patient is the most complete patient found in the original dataset; the raw values were not edited. If a few `NaN` values remain, they reflect tests that were not recorded. Missing values are handled later by the preprocessing pipeline using training-only medians and missing-value indicators.